# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Citation: {getattr(metadata, 'citeAs', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Review available record sets (tables), their `@id` values, fields (`@id`), and columns. This is necessary to understand what tabular data is available for analysis.

In [ ]:
from pprint import pprint

# Examine all RecordSets and their @id fields
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata. Exiting.")
else:
    print(f"Record sets found (each referenced by @id):\n")
    for record_set in record_sets:
        rset_id = getattr(record_set, '@id', None)
        rset_name = getattr(record_set, 'name', None)
        print(f"- RecordSet @id: {rset_id} | name: {rset_name}")
        fields = getattr(record_set, 'field', [])
        if fields:
            print("  - Fields (@id):")
            for field in fields:
                fid = getattr(field, '@id', None)
                fname = getattr(field, 'name', None)
                dtype = getattr(field, 'dataType', None)
                print(f"    - {fid} | name: {fname} | type: {dtype}")
        print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Select the record set `@id` of interest (e.g., the one containing main protein data) and list its available columns.

In [ ]:
# Replace this with the actual record set @id. Fetch dynamically if only one recordSet exists.
if not record_sets:
    raise ValueError("No record sets in the dataset! Cannot proceed.")
# For this dataset, let us assume the main table is the first one. Adjust as needed:
main_recordset_id = getattr(record_sets[0], '@id', None)

print(f"Using record set with @id: {main_recordset_id}")

# Load data to DataFrame
records = list(dataset.records(record_set=main_recordset_id))
df = pd.DataFrame(records)

print(f"Columns found in this record set:")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. The following cells demonstrate examples.

In [ ]:
# Choose a numeric field and a field to group by, based on the columns you printed above.
# For demonstration, let's select columns named 'mw' (molecular weight) and 'accession' if available.
numeric_field = None
group_field = None

# Try likely fields. Adjust manually based on your EDA above.
for col in df.columns:
    if col.lower() in ['mw', 'molecular_weight', 'molecularweight']:
        numeric_field = col
    if col.lower() in ['accession', 'protein_id', 'id']:
        group_field = col

if numeric_field is None:
    print("No obvious numeric field for analysis. Using the first float/integer column found.")
    for col in df.select_dtypes(['float', 'int']).columns:
        numeric_field = col
        break

if numeric_field is None:
    raise ValueError("No numeric columns available in this record set.")

print(f"Using numeric field: {numeric_field}")
if group_field:
    print(f"Using group field: {group_field}")

# Example: Filter where molecular weight (mw) > threshold (e.g., 50000)
threshold = 50000

# Only keep rows with valid numeric values
filtered_df = df[df[numeric_field].apply(pd.api.types.is_number)].copy()
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric column
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# If there is a group_field, group by it
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., molecular weight) and explore relationships in the filtered data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If group_field exists, plot group means (top 15 most common)
if group_field and group_field in df.columns:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:15]
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_means.index, y=group_means.values)
    plt.xticks(rotation=90)
    plt.ylabel(f"Mean {numeric_field}")
    plt.title(f"Top 15 groups by mean {numeric_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load, explore, and process the FAIR^2 mass spectrometry protein dataset. We loaded available record sets, fields, and visualized distributions of key quantitative features. You can further explore the dataset by analyzing other available fields or exporting processed data for downstream tasks.